# Week 3 - Day 4: MCP and the Study Assistant (project)

> **Student notebook (the Week 3 project).** A TODO cell with hints, then a check that prints `PASS` or `NOT YET`.

Today has three parts:
1. **MCP** (Model Context Protocol): a standard way to plug **ready-made tools** into any agent. You connect the official **LangChain docs** server, so your agent can read the docs.
2. **The project: a study assistant** that answers from the **course notes** (Day 3), **remembers** each student (Day 2), keeps its **context small** (Day 1), and reads the **official docs** (MCP).
3. **Evaluation**: is it more correct than a plain model?

The MCP code follows the docs page **[Model Context Protocol](https://docs.langchain.com/oss/python/langchain/mcp)**.

## Setup
Today you need **two** free keys:
- **Groq** (console.groq.com -> API Keys), saved as `GROQ_API_KEY`
- **Google AI Studio** (aistudio.google.com -> Get API key), saved as `GOOGLE_API_KEY`. It is free, and we use it only for embeddings

**On Google Colab:** click the key icon (Secrets) on the left, add the keys, and switch **Notebook access** on. **On your computer:** put them in a `.env` file.

> **Tips.** The free Groq plan allows about 8,000 tokens a minute. If a cell pauses, it is waiting for the limit. If a cell ends with `RateLimitError`, wait a minute and run it again. If a check ends with `NameError`, you have not run the TODO cell above it yet. Google's free plan allows 100 embedding requests a minute, and every chunk is one request: if you see `RESOURCE_EXHAUSTED`, wait a minute and run the cell again.

In [ ]:
%pip install -qU "langchain[mcp]" langgraph langchain-groq langchain-google-genai langchain-text-splitters python-dotenv requests

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()                                                   # on your computer: reads the keys from a .env file
try:
    from google.colab import userdata                           # on Google Colab: reads the keys from Secrets
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass

print("GROQ_API_KEY found:", "GROQ_API_KEY" in os.environ)
print("GOOGLE_API_KEY found:", "GOOGLE_API_KEY" in os.environ)

In [ ]:
from langchain.chat_models import init_chat_model

# One model for the whole week. max_retries=10: when the free plan's speed limit is reached, wait and try again.
llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0, max_retries=10)

from dataclasses import dataclass
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

---
# Part 1 - MCP

Every tool you wrote so far lives **inside your notebook**. **MCP** lets a **server** offer tools to **any** agent: the docs team runs one server, and every app can use its tools. It is like a USB plug for tools.

LangChain has a built-in MCP client, `MCPAdapter`. It is new, so it prints a **beta** warning. You can ignore it.

**One new idea: `await`.** MCP tools talk to a server over the internet, so they are **asynchronous**: you `await` them, and the agent runs with `await agent.ainvoke(...)` instead of `agent.invoke(...)`. In Colab you can write `await` directly in a cell.

In [ ]:
# TODO 1: connect to the LangChain docs server, and list its tools.
#   from langchain.mcp import MCPAdapter
#
#   async with MCPAdapter("https://docs.langchain.com/mcp") as adapter:      (a public server: no key needed)
#       docs_tools = await adapter.list_tools()
#
#   for docs_tool in docs_tools:
#       print(docs_tool.name, "-", docs_tool.description[:80])

The server offers three tools. One of them, `submit_feedback`, **sends a message to the LangChain team**: an action in the outside world. Give an agent **only the tools it needs**. We keep only the search tool.

In [ ]:
# CHECK 1 (and keep only the search tool)
docs_search = None
for docs_tool in docs_tools:
    if docs_tool.name == "search_docs_by_lang_chain":
        docs_search = docs_tool

result = await docs_search.ainvoke({"query": "short-term memory checkpointer"})
print("One search returned", len(str(result)), "characters")

if docs_search is not None and len(str(result)) > 1000:
    print("PASS: you are connected to the docs server.")
else:
    print("NOT YET: check the MCPAdapter cell.")

One search returns **tens of thousands of characters**: more than the free Groq plan's 8,000 tokens a minute. So, before the agent can use it, we **cut long tool results**, with a `@wrap_tool_call` middleware like the guard of Day 1. Because MCP tools are asynchronous, this middleware is too: the same code with `async def` and `await`.

In [ ]:
# TODO 2: a middleware that keeps only the first 3,000 characters of every tool result.
#   from langchain.agents.middleware import wrap_tool_call
#
#   @wrap_tool_call
#   async def shorten_tool_results(request, handler):
#       result = await handler(request)                     (run the tool)
#       result.content = str(result.content)[:3000]
#       return result

In [ ]:
# TODO 3: an agent that answers from the official docs. Note: await and ainvoke.
#   docs_agent = create_agent(
#       llm,
#       tools=[docs_search],
#       middleware=[shorten_tool_results],
#       system_prompt="Answer questions about LangChain. Call the search_docs_by_lang_chain tool once, then answer in 4 sentences and give the link.",
#   )
#   docs_answer = await docs_agent.ainvoke({"messages": "How do I add short-term memory to a LangChain agent?"})
#   print(docs_answer["messages"][-1].content)

In [ ]:
# CHECK 2
tool_sizes = []
for message in docs_answer["messages"]:
    if message.type == "tool":
        tool_sizes.append(len(str(message.content)))
print("Tool result sizes the model received:", tool_sizes)

if "checkpointer" in docs_answer["messages"][-1].content.lower() and tool_sizes and max(tool_sizes) <= 3000:
    print("PASS: the agent read the docs (cut to 3,000 characters) and explained the checkpointer.")
else:
    print("NOT YET: the answer should mention a checkpointer, and every tool result should be at most 3,000 characters.")

---
# Part 2 - The project: a study assistant

| Day | What the assistant gets | Block |
|---|---|---|
| 1 | a small context | `SummarizationMiddleware`, `shorten_tool_results` |
| 2 | a memory of each student | a store, `save_memory`, a dynamic prompt |
| 3 | the course notes | `search_course_notes` |
| 4 | the official docs | the MCP tool `docs_search` |

The pieces from Days 2 and 3 are given below, so you do not retype them. **You** join them into one agent.

In [ ]:
# Given (Day 3): the course notes in a vector store, and the search tool.
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

text = requests.get("https://raw.githubusercontent.com/AjaySharma78/Advgenai/main/week-3/course_notes.md").text
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100).split_documents([Document(page_content=text, metadata={"source": "course_notes.md"})])
vector_store = InMemoryVectorStore(GoogleGenerativeAIEmbeddings(model="gemini-embedding-001"))
vector_store.add_documents(chunks)

@tool
def search_course_notes(query: str) -> str:
    """Search the notes of the Advanced Generative AI course (Weeks 1 and 2). Use it for any question about the course."""
    texts = []
    for doc in vector_store.similarity_search(query, k=3):
        texts.append(doc.page_content)
    return "\n\n".join(texts)

print(len(chunks), "chunks of course notes are ready.")

In [ ]:
# Given (Day 2): who is talking, a store, a memory tool, and a dynamic prompt.
@dataclass
class Context:
    user_id: str

store = InMemoryStore()

@tool
def save_memory(fact: str, runtime: ToolRuntime[Context]) -> str:
    """Save one short fact about the student, for example a topic they find hard or how they like to learn."""
    runtime.store.put((runtime.context.user_id, "memories"), fact, {"fact": fact})
    return "Saved: " + fact

from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def study_prompt(request):
    facts = ""
    for item in request.runtime.store.search((request.runtime.context.user_id, "memories")):
        facts = facts + "- " + item.value["fact"] + "\n"
    return ("You are the study assistant of the Advanced Generative AI course.\n"
            "- For questions about the course, call search_course_notes and answer from the notes.\n"
            "- For questions about how to write LangChain code, call search_docs_by_lang_chain.\n"
            "- When the student tells you something about themselves, save it with save_memory.\n"
            "Answer in at most 120 words.\n"
            "What you know about this student:\n" + facts)

In [ ]:
# TODO 4: the study assistant: every piece of the week in one agent.
#   from langchain.agents.middleware import SummarizationMiddleware
#
#   study_agent = create_agent(
#       llm,
#       tools=[search_course_notes, save_memory, docs_search],
#       middleware=[study_prompt, shorten_tool_results, SummarizationMiddleware(model=llm, trigger=("tokens", 3000), keep=("messages", 6))],
#       store=store,
#       context_schema=Context,
#       checkpointer=InMemorySaver(),
#   )

In [ ]:
# Given: two conversations with Ram, in two different threads.
def tools_used(result):
    names = []
    for message in result["messages"]:
        if message.type == "tool":
            names.append(message.name)
    return names

ram = Context(user_id="ram")

first = await study_agent.ainvoke({"messages": "Hi, I'm Ram. Please remember that I find reducers confusing. What is a reducer in our course?"},
                                  {"configurable": {"thread_id": "ram-1"}}, context=ram)
print("Thread 1:", first["messages"][-1].content)
print("Tools   :", tools_used(first))
print()

second = await study_agent.ainvoke({"messages": "What topic did I say I find confusing? And how do I add a checkpointer to an agent, according to the official docs?"},
                                   {"configurable": {"thread_id": "ram-2"}}, context=ram)
print("Thread 2:", second["messages"][-1].content)
print("Tools   :", tools_used(second))

In [ ]:
# CHECK 3
saved = ""
for item in store.search(("ram", "memories")):
    saved = saved + item.value["fact"].lower()

if "reducer" in saved:
    print("PASS: the assistant saved that Ram finds reducers confusing.")
else:
    print("NOT YET: the reducer fact should be in Ram's memories. Check save_memory in the tools.")
if "search_course_notes" in tools_used(first):
    print("PASS: it answered the course question from the notes.")
else:
    print("NOT YET: the first answer should use search_course_notes.")
if "reducer" in second["messages"][-1].content.lower() and "search_docs_by_lang_chain" in tools_used(second):
    print("PASS: in a new thread it remembered the topic, and it read the official docs.")
else:
    print("NOT YET: the second answer should mention reducers and use the docs tool.")

---
# Part 3 - Evaluation: more correct than a plain model?

An **eval set** of questions about the course, each with the **right answer** from the notes. A **judge** compares each answer with the right answer. We test the study assistant and a plain model (no notes), and count the tokens.

In [ ]:
# TODO 5: add one question of your own, with its right answer from course_notes.md.
#   EVAL_SET = [
#       ("After how many attempts does the evaluator loop stop?", "After 3 attempts."),
#       ("What are the five levels of the autonomy ladder?", "L0 plain code, L1 LLM workflow, L2 tool agent, L3 multi-agent, L4 long-running."),
#       ("How many tokens a minute does the free Groq plan allow?", "About 8,000 tokens a minute."),
#       ("...", "..."),       (your own question and its right answer)
#   ]

In [ ]:
# TODO 6: the judge.
#   class Correct(BaseModel):
#       correct: bool = Field(description="True if the answer says the same thing as the right answer")
#
#   judge = llm.with_structured_output(Correct, method="json_schema", strict=True).with_retry(stop_after_attempt=3)

In [ ]:
# Given: run both on the eval set. It takes about a minute.
from langchain_core.callbacks import get_usage_metadata_callback

assistant_right = 0
plain_right = 0
assistant_tokens = 0
plain_tokens = 0
eval_user = Context(user_id="eval")

for question, right_answer in EVAL_SET:
    with get_usage_metadata_callback() as cb:
        result = await study_agent.ainvoke({"messages": question}, {"configurable": {"thread_id": question}}, context=eval_user)     # a new thread for each question
    assistant_answer = result["messages"][-1].content
    for usage in cb.usage_metadata.values():
        assistant_tokens = assistant_tokens + usage["total_tokens"]

    with get_usage_metadata_callback() as cb:
        plain_answer = llm.invoke("Answer in one sentence. In the Advanced Generative AI course: " + question).content
    for usage in cb.usage_metadata.values():
        plain_tokens = plain_tokens + usage["total_tokens"]

    if judge.invoke(f"Question: {question}\nRight answer: {right_answer}\nAnswer: {assistant_answer}").correct:
        assistant_right = assistant_right + 1
    if judge.invoke(f"Question: {question}\nRight answer: {right_answer}\nAnswer: {plain_answer}").correct:
        plain_right = plain_right + 1
    print("Q:", question)
    print("   assistant:", assistant_answer[:120].replace("\n", " "))
    print("   plain    :", plain_answer[:120].replace("\n", " "))

print()
print("                  right answers   tokens")
print("study assistant ", assistant_right, "of", len(EVAL_SET), "        ", assistant_tokens)
print("plain model     ", plain_right, "of", len(EVAL_SET), "        ", plain_tokens)

In [ ]:
# TODO 7: read your table and decide.
#   conclusion = "..."     (two sentences with your numbers: how much more correct was the assistant, and was it worth its tokens?)

In [ ]:
# CHECK 4
has_number = False
for character in conclusion:
    if character.isdigit():
        has_number = True

if assistant_right >= plain_right and len(conclusion.split()) >= 20 and has_number:
    print("PASS: a decision with numbers.")
else:
    print("NOT YET: write two sentences that use your numbers.")

---
# Week 3 in one table

| Day | Idea | Blocks |
|---|---|---|
| 1 | context engineering | a checkpointer and thread id, `SummarizationMiddleware`, `ContextEditingMiddleware`, a `@wrap_tool_call` guard |
| 2 | long-term memory | `InMemoryStore`, namespaces, `ToolRuntime`, `context_schema`, `@dynamic_prompt` |
| 3 | agentic RAG | `RecursiveCharacterTextSplitter`, `GoogleGenerativeAIEmbeddings`, `InMemoryVectorStore`, the retriever as a tool |
| 4 | MCP | `MCPAdapter`, `await adapter.list_tools()`, `await agent.ainvoke(...)` |

**The assignment** finishes the week: chunk sizes, a forget tool, your own MCP server, a poisoned document, and a short memo.